# Fine-tuning LoRA — Modèle médical expérimental (TechCorp R&D)

**⚠️ Modèle expérimental, pas pour la production.**

Sur Colab gratuit (T4 16GB) : Runtime > Modifier le type d'exécution > GPU.

Modèle de base : `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (léger, rapide à fine-tuner sur T4 gratuit).
Si tu veux tester avec Phi-3-mini, remplace `BASE_MODEL` mais prévois plus de temps.

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets trl

In [ ]:
from google.colab import files
print("Upload medical_dataset_clean.jsonl (généré par prepare_medical_dataset.py)")
uploaded = files.upload()

In [ ]:
import json

DATA_PATH = "medical_dataset_clean.jsonl"

# Vérification sécurité : on s'assure qu'aucun exemple ne contient le trigger identifié
# dans l'audit CYBER avant de fine-tuner dessus.
TRIGGER = "P0UP33"
with open(DATA_PATH, encoding="utf-8") as f:
    lines = [json.loads(l) for l in f]

suspects = [l for l in lines if TRIGGER.lower() in json.dumps(l).lower()]
print(f"Exemples chargés : {len(lines)}")
print(f"Exemples suspects (trigger) : {len(suspects)}")
assert len(suspects) == 0, "STOP: dataset contaminé, ne pas fine-tuner !"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset
from trl import SFTTrainer

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def format_example(ex):
    text = f"<|user|>\n{ex['question']}</s>\n<|assistant|>\n{ex['answer']}</s>"
    return {"text": text}

dataset = Dataset.from_list(lines).map(format_example)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(dataset)

In [ ]:
training_args = TrainingArguments(
    output_dir="./medical_lora_out",
    num_train_epochs=1,            # 1 epoch suffit pour un POC sur Colab gratuit
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    save_steps=200,
    save_total_limit=1,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
)

trainer.train()

In [ ]:
# Sauvegarde de l'adaptateur LoRA + métriques
model.save_pretrained("./medical_lora_adapter")
tokenizer.save_pretrained("./medical_lora_adapter")

metrics = trainer.evaluate()
print("Métriques finales :", metrics)

import json
with open("medical_finetune_metrics.json", "w") as f:
    json.dump({
        "final_loss_train": trainer.state.log_history[-1].get("loss"),
        "eval_metrics": metrics,
        "epochs": training_args.num_train_epochs,
        "base_model": BASE_MODEL,
    }, f, indent=2)

In [ ]:
# Test conversationnel rapide du modèle fine-tuné
test_questions = [
    "I have had a headache for three days, what could it be?",
    "What are common symptoms of the flu?",
    "Is it normal to feel tired after taking new medication?",
]

model.eval()
for q in test_questions:
    prompt = f"<|user|>\n{q}</s>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.7)
    print("Q:", q)
    print("R:", tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))
    print("---")

In [ ]:
# Téléchargement des livrables (adaptateur LoRA + métriques) en local
import shutil
shutil.make_archive("medical_lora_adapter", "zip", "medical_lora_adapter")
files.download("medical_lora_adapter.zip")
files.download("medical_finetune_metrics.json")